# Stage 2 — Stenosis detection (Colab/Kaggle BUILD)

**Thin notebook** — all logic is in importable `src/*`. This just: env setup → prep → `src.train.train_detector.train` on GPU → export. YOLO stack; CoreML export is one Ultralytics call on the Mac.

Floor: **F1 > 0.57** on ARCADE stenosis. Runs on the free Colab **T4** or a Kaggle **P100/T4** GPU runtime — do NOT run on CPU.

**Faster, same quality:** speed knobs live in `configs/stenosis_yolo.yaml` → `train: {cache, workers, patience, amp}` — RAM/disk dataset cache, mixed precision, and early-stop once the val metric plateaus (keeps the same `best.pt`, skips wasted epochs). cuDNN autotune is enabled in the setup cell. Optional: cold-start SSL with an open-vocab **Grounding DINO** pass by setting `ssl.seed: gdino` (needs `transformers`).

In [ ]:
# 1) Get repo + install, then detect environment (Colab Drive / Kaggle / local)
import os, sys
REPO = '/content/drive/MyDrive/intv-img/interventional-imaging-pipeline'  # colab; on kaggle: /kaggle/working/...
if not os.path.exists(REPO):
    !git clone <YOUR_REPO_URL> {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install ultralytics>=8.2 pycocotools opencv-python pyyaml
# optional — only if you set ssl.seed: gdino below:
# !pip -q install transformers
from src import env
E = env.setup()          # mounts Drive on Colab; sets nnU-Net paths; prints device
assert E['device'] == 'cuda', 'Switch runtime to GPU (Runtime > Change runtime type).'

# speed (quality-neutral): let cuDNN autotune conv algos for the fixed imgsz input
import torch
torch.backends.cudnn.benchmark = True

## Data
Place **ARCADE** (task-2 stenosis, COCO) and **Danilov** under `data/raw/`. On Kaggle, attach them as Datasets (they appear under `/kaggle/input/`) and symlink into `data/raw/`.

In [ ]:
# 2) Standardize -> YOLO (images/labels/{train,val} + data.yaml). Importable converter.
import yaml
from src.data_prep import danilov_to_yolo
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
danilov_to_yolo.main(cfg)

In [ ]:
# 3) Train YOLO11n (+ pseudo-label SSL if enabled) on GPU. Heavy lifting lives in src.
#    Speed comes from configs/stenosis_yolo.yaml -> train: {cache, workers, patience, amp}.
#    Open-vocab cold start: set ssl.seed: gdino in that config (needs transformers) to have
#    Grounding DINO label the unlabeled frames before self-training.
from src.train.train_detector import train, train_kwargs
print('speed/train kwargs ->', train_kwargs(cfg), '| ssl.seed:', cfg.get('ssl', {}).get('seed'))
best = train(cfg, project=f"{E['runs']}/stenosis")
print('best weights ->', best)

## Export handoff
Save `best.pt` to Drive/working. On the **Mac**: `python -m src.export.yolo_to_coreml --weights best.pt` (one Ultralytics call, NMS baked in) → `.mlpackage`, then run `src.serve.realtime --task det`.

In [ ]:
# 4) (optional) sanity-export to CoreML here if on macOS runner; usually done on the Mac.
print('Handoff: copy', best, 'to the Mac, then: make export-coreml-yolo MODEL=', best)